# 19.5 Influence functions

Influence estimates how much one training point would move a prediction if it were upweighted or removed.

This is a gap topic in the plan, so the notebook keeps the verified toy numbers explicit and uses a small damped Hessian scaffold for the ladder.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

Influence uses the second-order approximation $I_i(z)=-
abla_	heta \ell(z)^	op H_	heta^{-1}
abla_	heta \ell(z_i)$. Damping keeps the inverse Hessian from exploding.

In [ ]:

def influence_scores(test_grad, train_grads, hessian, damping=0.1):
    damped = hessian + damping * np.eye(hessian.shape[0])
    solved = np.linalg.solve(damped, train_grads.T)
    scores = -test_grad @ solved
    return scores

test_grad_toy = np.array([1.0, 0.0, 0.0])
train_grads_toy = np.array([
    [-0.5, 0.0, 0.0],
    [0.7, 0.0, 0.0],
    [-0.2, 0.0, 0.0],
])
hessian_toy = np.eye(3)
scores_toy = influence_scores(test_grad_toy, train_grads_toy, hessian_toy, damping=0.0)
abs_toy = float(np.abs(scores_toy).sum())
share_toy = float(abs(scores_toy[0]) / abs_toy)
guarded_toy = float(scores_toy.sum() + 0.8 * abs_toy)
contrast_toy = float(scores_toy.sum() - scores_toy[2])

assert np.allclose(scores_toy, [0.5, -0.7, 0.2])
assert np.isclose(scores_toy.sum(), 0.0)
assert np.isclose(abs_toy, 1.4)
assert np.isclose(round(share_toy, 3), 0.357)
assert np.isclose(guarded_toy, 1.12)
assert np.isclose(contrast_toy - scores_toy.sum(), -0.2)

print("influence scores", scores_toy)
print("absolute mass", abs_toy)
print("top share", round(share_toy, 3))


For the ladder, we fit a binary one-vs-rest logistic model on a tiny standardized subset, compute per-example gradients and a damped Hessian, then compare influence ranks with actual leave-one-out score changes.

In [ ]:

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_gradients(X, y, coef):
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    logits = X_aug @ coef
    p = sigmoid(logits)
    grads = (p - y)[:, None] * X_aug
    return grads, p, X_aug


def fit_binary_logistic_subset(x_tr, y_tr, positive_class, max_points=60):
    keep = np.arange(min(max_points, len(y_tr)))
    X_small = x_tr[keep]
    y_small = (y_tr[keep] == positive_class).astype(float)
    if len(np.unique(y_small)) < 2:
        y_small[0] = 1.0 - y_small[0]
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_small, y_small)
    coef = np.r_[clf.intercept_[0], clf.coef_[0]]
    return clf, coef, X_small, y_small


def influence_vs_loo(x_tr, y_tr, x_test, positive_class, damping=0.1):
    clf, coef, X_small, y_small = fit_binary_logistic_subset(x_tr, y_tr, positive_class)
    train_grads, p, X_aug = logistic_gradients(X_small, y_small, coef)
    weights = p * (1.0 - p)
    hessian = (X_aug.T * weights) @ X_aug / len(y_small)
    test_y = np.array([1.0])
    test_grads, test_p, test_aug = logistic_gradients(x_test.reshape(1, -1), test_y, coef)
    scores = influence_scores(test_grads[0], train_grads, hessian, damping=damping)
    base_score = float(clf.predict_proba(x_test.reshape(1, -1))[0, 1])
    loo_changes = []
    for i in range(min(20, len(y_small))):
        mask = np.ones(len(y_small), dtype=bool)
        mask[i] = False
        if len(np.unique(y_small[mask])) < 2:
            loo_changes.append(0.0)
            continue
        refit = LogisticRegression(max_iter=2000)
        refit.fit(X_small[mask], y_small[mask])
        new_score = float(refit.predict_proba(x_test.reshape(1, -1))[0, 1])
        loo_changes.append(new_score - base_score)
    corr = rank_corr(scores[: len(loo_changes)], loo_changes)
    return corr, scores, np.array(loo_changes)


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is rank correlation between damped influence scores and actual leave-one-out retraining effects on tiny subsets.

In [ ]:

infl_rows = []
infl_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    positive_class = int(np.unique(y_tr)[-1])
    corr, scores, loo_changes = influence_vs_loo(x_tr, y_tr, x_te[0], positive_class, damping=0.2)
    infl_rows.append({"rung": rung, "name": name, "rank_corr": corr, "max_abs_influence": float(np.max(np.abs(scores)))})
    infl_artifacts.append((name, scores, loo_changes))

infl_table = pd.DataFrame(infl_rows)
print(infl_table.to_string(index=False))


## Results visualization

Panels compare predicted influence and leave-one-out changes for the audited subset. The curve shows rank-correlation-vs-stress.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], infl_artifacts):
    name, scores, loo_changes = artifact
    keep = min(20, len(loo_changes))
    ax.plot(np.arange(keep), scores[:keep], marker="o", label="influence")
    ax.plot(np.arange(keep), loo_changes[:keep], marker="x", label="leave-one-out")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(name[:26])
    ax.legend(fontsize=8)

axes[5].plot(infl_table["rung"], infl_table["rank_corr"], marker="o")
axes[5].set_title("correlation vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("rank correlation")
axes[5].set_ylim(-1.05, 1.05)
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: ignoring damping makes $H^{-1}$ unstable. The fix is a damping grid plus a leave-one-out sanity check.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
positive_class = int(np.unique(y_tr)[-1])
for damping in [0.0, 0.01, 0.1, 1.0]:
    try:
        corr, scores, loo_changes = influence_vs_loo(x_tr, y_tr, x_te[0], positive_class, damping=damping)
        max_score = float(np.max(np.abs(scores)))
        print("damping", damping, "rank_corr", corr, "max_abs", max_score)
    except np.linalg.LinAlgError:
        print("damping", damping, "failed because Hessian was singular")


## Evaluate it + Practice

- Compare the reported influence/leave-one-out rank correlation with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: set damping to zero and watch for singular solves or unstable max influence
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.